# NeuroAssist fine-tune 3/5 — Document Q&A (answer or say "not there")

Fine-tunes **Qwen2.5-3B-Instruct** to answer questions about a document in the app's exact JSON format `{found, quote, answer}`, and to say `found: false` when the document doesn't contain the answer. The app then checks the quote really is in the document.

**Data:** SQuAD 2.0 **train** (the practice documents come from the *dev* split, so there's no overlap). Each document is the question's paragraph plus a few unrelated paragraphs, to match the practice pack's multi-paragraph documents. **Scored on:** held-out SQuAD train articles not used for training.

Ask the organizers if SQuAD *train* is OK, since the practice documents come from SQuAD *dev*.

**Run:** `Runtime → Change runtime type → A100 GPU` (T4 works, slower), then `Runtime → Run all`. Only the ⚙️ Settings cell is meant to be edited.

⚠️ **Hackathon rule:** train only on TRAIN splits. Every practice item comes from a validation/test split, and the judges' items almost certainly do too. Never train on the practice pack.

## 1. Install

In [ ]:
# Unsloth installs matching torch / transformers / trl / peft / bitsandbytes.
# If a new release breaks, pin the previous one, e.g. unsloth==2026.11.6
!pip install -q unsloth

## 2. ⚙️ Settings

In [ ]:
MODEL_NAME  = "unsloth/Qwen2.5-3B-Instruct"
RUN_NAME    = "neuro-doc-qa"
OLLAMA_NAME = "neuro-doc-qa"
APP_SETTING = "NEUROASSIST_TEXT_MODEL"

# Copied from the app (neuroassist/tools/text_qa.py). Must match exactly.
SYSTEM_TEMPLATE = 'You answer questions using ONLY the {source} you are given. Never use outside knowledge. If the {source} does not contain the answer, say so.'
PROMPT_TEMPLATE = '{Source}:\n<<<\n{doc}\n>>>\n\nQuestion: {question}\n\nIf the {source} answers the question: set "found" to true, copy the exact sentence from the {source} that contains the answer into "quote", and put a short answer (a few words) in "answer".\nIf it does not: set "found" to false and leave "quote" and "answer" empty.\nCheck carefully: names, numbers and dates in the question must match the {source} exactly.'
SYSTEM_PROMPT = SYSTEM_TEMPLATE.format(source="document")

TRAIN_ROWS           = 8000
UNANSWERABLE_SHARE   = 0.40     # practice pack: 5 of 20
EXTRA_PARAGRAPHS     = (0, 3)   # unrelated paragraphs added around the real one
HELDOUT_ARTICLES     = 0.05     # share of SQuAD train articles kept out for scoring
EVAL_ROWS            = 300

MAX_SEQ_LENGTH = 3072
LORA_RANK, LORA_ALPHA = 16, 16
EPOCHS, MAX_STEPS = 1, None
LEARNING_RATE = 2e-4
BATCH_SIZE, GRAD_ACCUM = 8, 2
SEED = 3407
SAVE_TO_DRIVE, DRIVE_DIR = True, "/content/drive/MyDrive/neuroassist-finetunes"
GGUF_QUANT = "q4_k_m"

## 3. GPU and Google Drive

In [ ]:
import torch, os, json, random, re
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, {gpu.total_memory / 2**30:.1f} GB")
if gpu.total_memory < 20 * 2**30 and BATCH_SIZE > 2:      # T4 / L4: keep the effective batch size
    GRAD_ACCUM, BATCH_SIZE = GRAD_ACCUM * BATCH_SIZE // 2, 2
    print(f"Small GPU: BATCH_SIZE=2, GRAD_ACCUM={GRAD_ACCUM}")
random.seed(SEED)

OUT_DIR = f"/content/{RUN_NAME}"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")      # mounted first, so nothing is lost if a later cell fails
    OUT_DIR = f"{DRIVE_DIR}/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving to", OUT_DIR)

## 4. Load the model and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel, is_bf16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()


def chat(user, assistant=None, system=None):
    """One training/eval conversation in the app's format."""
    msgs = [{"role": "system", "content": system if system is not None else SYSTEM_PROMPT},
            {"role": "user", "content": user}]
    if assistant is not None:
        msgs.append({"role": "assistant", "content": assistant})
    return msgs


def generate_batch(conversations, max_new_tokens=256, batch_size=16):
    """Greedy answers for a list of conversations (without the assistant turn)."""
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    outs = []
    for i in range(0, len(conversations), batch_size):
        prompts = [tokenizer.apply_chat_template(c, add_generation_prompt=True, tokenize=False)
                   for c in conversations[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = "right"
    FastLanguageModel.for_training(model)
    return [o.strip() for o in outs]

## 5. Build the training data and the held-out test

In [ ]:
from collections import defaultdict
from datasets import load_dataset

squad = load_dataset("rajpurkar/squad_v2", split="train")
titles = sorted(set(squad["title"]))
random.shuffle(titles)
heldout = set(titles[: max(1, int(len(titles) * HELDOUT_ARTICLES))])
paragraphs = sorted(set(squad["context"]))

SENT = re.compile(r"[^.!?]+(?:[.!?]+[\"')\]]*)?\s*")
def sentence_at(context, start):
    for m in SENT.finditer(context):
        if m.start() <= start < m.end():
            return m.group(0).strip()
    return None

def build(row):
    """(conversation, target dict) in the app's format, or None if unusable."""
    k = random.randint(*EXTRA_PARAGRAPHS)
    others = [p for p in random.sample(paragraphs, k + 3) if p != row["context"]][:k]
    paras = others + [row["context"]]
    random.shuffle(paras)
    doc = "\n\n".join(paras)
    if row["answers"]["text"]:
        ans, start = row["answers"]["text"][0], row["answers"]["answer_start"][0]
        quote = sentence_at(row["context"], start)
        if not quote or ans not in quote:
            return None
        target = {"found": True, "quote": quote, "answer": ans}
    else:
        target = {"found": False, "quote": "", "answer": ""}
    user = PROMPT_TEMPLATE.format(Source="Document", source="document", doc=doc, question=row["question"])
    return user, target, doc

def sample(rows, n):
    ans = [r for r in rows if r["answers"]["text"]]
    no = [r for r in rows if not r["answers"]["text"]]
    n_no = int(n * UNANSWERABLE_SHARE)
    picked = random.sample(no, min(n_no, len(no))) + random.sample(ans, min(n - n_no, len(ans)))
    random.shuffle(picked)
    return picked

train_rows = [r for r in squad if r["title"] not in heldout]
test_rows = [r for r in squad if r["title"] in heldout]
examples = []
for r in sample(train_rows, TRAIN_ROWS):
    b = build(r)
    if b:
        examples.append(chat(b[0], json.dumps(b[1], ensure_ascii=False)))
print(f"{len(examples)} training examples · {len(heldout)} held-out articles")

eval_set = [b for b in (build(r) for r in sample(test_rows, EVAL_ROWS)) if b]

def _norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", s.lower())).strip()

def evaluate_model():
    outs = generate_batch([chat(u) for u, _, _ in eval_set], max_new_tokens=200)
    valid = ans_ok = ans_n = no_ok = no_n = quote_ok = 0
    for out, (_, gold, doc) in zip(outs, eval_set):
        try:
            p = json.loads(out); valid += 1
        except Exception:
            p = {}
        if gold["found"]:
            ans_n += 1
            hit = bool(p.get("found")) and _norm(gold["answer"]) in _norm(p.get("answer", "")) or \
                  (p.get("answer") and _norm(p["answer"]) in _norm(gold["answer"]))
            ans_ok += bool(hit)
            quote_ok += bool(p.get("quote")) and _norm(p["quote"]) in _norm(doc)
        else:
            no_n += 1
            no_ok += not p.get("found", False)
    return {"valid JSON": round(valid / len(outs), 3),
            "answerable correct": round(ans_ok / max(ans_n, 1), 3),
            "quote really in doc": round(quote_ok / max(ans_n, 1), 3),
            "unanswerable -> not found": round(no_ok / max(no_n, 1), 3)}

## 6. Format with the model's chat template

In [ ]:
from datasets import Dataset

texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in examples]
lengths = [len(tokenizer(t, add_special_tokens=False).input_ids) for t in texts]
keep = [t for t, l in zip(texts, lengths) if l <= MAX_SEQ_LENGTH]
print(f"{len(keep)} examples ({len(texts) - len(keep)} dropped as longer than {MAX_SEQ_LENGTH} tokens), "
      f"median {sorted(lengths)[len(lengths) // 2]} tokens")
data = Dataset.from_dict({"text": keep}).shuffle(seed=SEED)
split = data.train_test_split(test_size=min(0.02, 500 / len(data)), seed=SEED)
train_ds, val_ds = split["train"], split["test"]
print(f"train {len(train_ds)} · validation-loss set {len(val_ds)}")
print("\n--- one training example ---\n" + train_ds[0]["text"][:1500])

## 7. Score the base model (before)

In [ ]:
print("Scoring the base model before training (this is the number to beat)...")
before = evaluate_model()
print(json.dumps(before, indent=2))

## 8. Train

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

wanted = dict(
    output_dir=f"/content/checkpoints/{RUN_NAME}",
    dataset_text_field="text", max_length=MAX_SEQ_LENGTH, max_seq_length=MAX_SEQ_LENGTH,
    packing=False, dataset_num_proc=2,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS, max_steps=MAX_STEPS if MAX_STEPS else -1,
    learning_rate=LEARNING_RATE, warmup_ratio=0.05, lr_scheduler_type="cosine",
    optim="adamw_8bit", weight_decay=0.01,
    bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
    logging_steps=10, eval_strategy="steps", eval_steps=100, save_strategy="no",
    seed=SEED, report_to="none")
accepted = inspect.signature(SFTConfig.__init__).parameters      # TRL renames args between versions
if "max_length" in accepted:
    wanted.pop("max_seq_length")
args = SFTConfig(**{k: v for k, v in wanted.items() if k in accepted})

kw = dict(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=args)
kw["processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters
   else "tokenizer"] = tokenizer
trainer = SFTTrainer(**kw)

# Loss only on the assistant's answer. Turn markers are read from this model's chat template.
from unsloth.chat_templates import train_on_responses_only
probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "@@U@@"}, {"role": "assistant", "content": "@@A@@"}], tokenize=False)
def _marker(text, words):
    idx = max(text.rfind(w) for w in words)
    return text[text.rfind("<", 0, idx):]
user_part = _marker(probe[:probe.index("@@U@@")], ["user"])
asst_part = _marker(probe[probe.index("@@U@@") + 5:probe.index("@@A@@")], ["assistant", "model"])
print(f"training on answers only: {user_part!r} … {asst_part!r}")
trainer = train_on_responses_only(trainer, instruction_part=user_part, response_part=asst_part)

stats = trainer.train()
print(f"done in {stats.metrics['train_runtime'] / 60:.1f} min · train loss {stats.metrics['train_loss']:.3f}")

## 9. Score again (after) and compare

In [ ]:
after = evaluate_model()
print(f"{'metric':<28}{'before':>10}{'after':>10}")
for k in before:
    b, a = before[k], after[k]
    print(f"{k:<28}{b:>10.3f}{a:>10.3f}" if isinstance(b, float) else f"{k:<28}{b!s:>10}{a!s:>10}")
json.dump({"model": MODEL_NAME, "before": before, "after": after}, open(f"{OUT_DIR}/scores.json", "w"), indent=2)
print(f"\nKeep this fine-tune only if it also beats the stock model in the app's eval (eval/run_eval.py).")

## 10. Save adapters, export GGUF and a Modelfile for Ollama

In [ ]:
import glob, shutil

model.save_pretrained(f"{OUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUT_DIR}/lora")
print("LoRA adapters saved:", f"{OUT_DIR}/lora")

gguf_dir = f"/content/{RUN_NAME}_gguf"
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method=GGUF_QUANT)
found = [f for f in glob.glob(f"/content/{RUN_NAME}_gguf*/**/*", recursive=True)
         if f.endswith(".gguf") or os.path.basename(f) == "Modelfile"]
for f in found:
    shutil.copy(f, OUT_DIR)
    print("copied", os.path.basename(f), f"({os.path.getsize(f) / 2**20:.0f} MB)")
ggufs = sorted(os.path.basename(f) for f in found if f.endswith(".gguf"))
assert ggufs, "No .gguf was produced; check the export log above."
gguf = next((g for g in ggufs if GGUF_QUANT.lower() in g.lower()), ggufs[0])

# Our own Modelfile, pointing at the quantized file. Qwen models use the ChatML format.
assert "qwen" in MODEL_NAME.lower(), "Modelfile below is for Qwen; write one for this model family."
modelfile = f"""FROM ./{gguf}
TEMPLATE \"\"\"{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
\"\"\"
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER temperature 0
"""
open(f"{OUT_DIR}/Modelfile", "w").write(modelfile)
print(f"""
Done. On the demo laptop, copy {gguf} and Modelfile from Drive into one folder, then:
    ollama create {OLLAMA_NAME} -f Modelfile
and run the app / eval with:
    $env:{APP_SETTING}="{OLLAMA_NAME}"
""")